In [12]:
import random

In [22]:
from collections import Counter

In [ ]:
states = ['fresh', 'fill', 'bust', 'rolled']
actions = ['roll', 'hold', 'stop']

In [189]:
class Dice(object):
    
    def __init__(self, num_dice=6, max_score=1000):
        self.num_dice = num_dice
        self.max_score = max_score
        self.state = 'fresh'
        self.pointers = {5: 50, 1: 100}
        self.dice = None
        self.score = 0

    def play(self):
        still_playing = True
        dice_to_hold = []
        n = 0
        while still_playing:
            print(n)
            n += 1
            action = random.choice(self.get_actions_from_state())
            print(action)
            if action == self.hold:
                dice_to_hold = [idx for idx, die in enumerate(self.dice) if die in self.pointers]
                random.shuffle(dice_to_hold)
                dice_to_hold = dice_to_hold[:random.choice(range(len(dice_to_hold)))]
                state, score, still_playing = self.step(action, dice_to_hold)
                continue
            state, score, still_playing = self.step(action)
        print('Done!')
    
    def step(self, action, *args):
        """
        :action: A function
        """
        result = action(*args)
        self.state = self.determine_state()
        if self.state == 'bust':
            self.score = 0
            self.print_game()
            return self.state, self.score, True
        
        # If we didn't bust
        roll_score = result if action == self.hold else 0
        self.score += roll_score
        self.print_game()
        return self.state, self.score, False
        
    def roll(self):
        self.dice = [random.choice(range(1, self.num_dice + 1)) for _ in range(self.num_dice)]
    
    def stop(self):
        self.score = self.calc_score()
        self.state = 'stop'
        
    def hold(self, dice_idxs):
        held_dice = [self.dice[idx] for idx in dice_idxs]
        self.num_dice -= len(held_dice)
        return self.calc_score(held_dice)

    def calc_score(self, dice=None):
        dice = dice if dice else self.dice
        counts = Counter(dice)
        score = (
            100 * counts.get(1, 0) +
            50 * counts.get(5, 0)
        )
        return score

    def determine_state(self):
        if self.state == 'stop':
            return 'stop'
        if not any(die in self.pointers for die in self.dice):
            self.state = 'bust'
        elif all(die in self.pointers for die in self.dice):
            self.state = 'fill'
            # TODO: Add a points bonus incentive here
        else:
            self.state = 'rolled'
        return self.state

    def get_actions_from_state(self):
        return {
            'fresh': [self.roll],
            'rolled': [self.stop, self.hold],
            'fill': [self.stop, self.roll],
            'bust': [None],
            'stop': [None]
        }[self.state]
    
    def __str__(self):
        return f"""State: {self.state}\nDice: {self.dice}\nScore: {self.score}\nActions: {self.get_actions_from_state()}"""
    
#     def __repr__(self):
#         return self.__str__()
    
    def print_game(self):
        print(self.__str__())

In [196]:
game = Dice(num_dice=6)
game.print_game()

State: fresh
Dice: None
Score: 0
Actions: [<bound method Dice.roll of <__main__.Dice object at 0x10af72090>>]


In [202]:
game.play()

0
None


TypeError: 'NoneType' object is not callable

In [131]:
game.roll()

State: rolled
Dice: [1, 1, 3]
Score: 0
Actions: ['stop', 'hold']


In [134]:
game.stop()

TypeError: unhashable type: 'list'